# SKKB — Results Viewer

JIRA: **GCAI-3092** — diagnostic LLM-as-a-Judge evaluation of the SK (Slovak) Knowledge-Base chatbot.

Loads the judge checkpoint from [`skkb_001_baseline.ipynb`](skkb_001_baseline.ipynb), computes aggregates, validates the judge against the pre-existing expert 1–10 score, and extracts the product-improvement signals the judge emits.

**Judge-scored:** 8 dimensions (query_clarity, case_scope_fit, selection_semantic_relevance, selected_context_sufficiency, optimal_retrieved_context_adequacy, answer_expected_alignment, answer_groundedness, language_compliance).

**Derived programmatically** in this notebook (NOT asked of the judge):
- `retrieval_recall` / `retrieval_recall_score` — set operation on `expected_enums` ⊆ `retrieved_enum_ids`.
- `total_score_judge` — sum of the 8 dimension scores, 0–16.
- `selection_was_optimal` — exact set match between `reranked_enum_ids` and `expected_enums`.
- `root_cause_category` — pipeline-ordered decision tree on the scores plus missing-ENUM sets.

Renders per-dimension distributions, root-cause breakdown, retriever vs reranker miss attribution, KB-gap rate, language-compliance failures, judge-vs-expert Spearman, and the three improvement-suggestion excerpts. Exports an HTML report.


In [1]:
RUN_ON_DBX:bool = False

In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

In [ ]:
!python skkb_report.py --checkpoint checkpoints/databricks_gpt51/evals_skkb_exp_001_baseline_no_expected_enums_high.csv --output ../../../skkb_report.html

## Parameters

In [0]:
YAML_FILE_NAME = "skkb_exp_001_baseline_no_expected_enums"
REASONING_EFFORT = "high"
CHECKPOINT_SUFFIX = ""  # Set to "_test" to inspect the smaller test checkpoint.
MODEL_DIR = "databricks_gpt51"  # Subdir under checkpoints/ — e.g. "chat00", "databricks_gpt51".
CHECKPOINT_CSV = f"checkpoints/{MODEL_DIR}/evals_{YAML_FILE_NAME}_{REASONING_EFFORT}{CHECKPOINT_SUFFIX}.csv"

# if RUN_ON_DBX:
#     REPORT_DIR = f"../Workspace/Repos/Shared_HeyGeorge/ds-evals/reports/{YAML_FILE_NAME}"
# else:
#     REPORT_DIR = "../reports"
# REPORT_PATH = f"{REPORT_DIR}/report{CHECKPOINT_SUFFIX}.html"

# Dimension weights — must match the YAML rubric. 8 judge-scored dimensions, weight sum = 11.5.
# `retrieval_recall` is NOT asked of the judge; it's computed programmatically below
# from retrieved_enum_ids vs expected_enums.
DIMENSION_WEIGHTS = {
    "query_clarity":           1.0,
    "selection_semantic_relevance":        1.5,
    "selected_context_sufficiency":        1.5,
    "optimal_retrieved_context_adequacy":  1.5,
    "answer_expected_alignment":           2.0,
    "answer_groundedness":                 2.0,
    "language_compliance":     1.0,
}
PASS_THRESHOLD = 1.4


## Imports and load

In [0]:
import ast
import csv
import json
import os
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

# Raise CSV field-size limit — checkpoint rows contain large JSON blobs (>128 KB).
csv.field_size_limit(sys.maxsize)

# The notebook lives in experiments/skkb/notebooks/ both locally and on Databricks
# (/Workspace/Users/<user>/hg-ds-evals/experiments/skkb/notebooks), so cwd is the
# right anchor for sibling modules and checkpoints in either environment.
if (Path.cwd() / "skkb_checkpoint.py").exists():
    sys.path.insert(0, str(Path.cwd()))
from skkb_checkpoint import read_checkpoint_csv

def _resolve_checkpoint_path(path: str | Path) -> Path:
    path = Path(path)
    if path.is_absolute():
        candidates = [path]
    else:
        candidates = [path, Path.cwd() / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(f"Could not find checkpoint CSV. Searched:\n{searched}")


checkpoint_path = _resolve_checkpoint_path(CHECKPOINT_CSV)
if not RUN_ON_DBX:
    repo_candidates = [p for p in checkpoint_path.parents if p.name == "hg-ds-evals"]
    repo_root = repo_candidates[0] if repo_candidates else checkpoint_path.parent
    REPORT_DIR = str(repo_root / "reports" / YAML_FILE_NAME)
    REPORT_PATH = f"{REPORT_DIR}/report{CHECKPOINT_SUFFIX}.html"
df = read_checkpoint_csv(checkpoint_path)

print(f"checkpoint: {checkpoint_path}")
print(f"rows: {len(df)}   cols: {len(df.columns)}")
print("columns:", list(df.columns))

checkpoint: /Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/notebooks/checkpoints/databricks_gpt51/evals_skkb_exp_001_baseline_no_expected_enums_high.csv
rows: 484   cols: 89
columns: ['test_case_id', 'trace_id', 'user_query', 'reranked_kb_context', 'reranked_enum_ids', 'raw_vector_db_retrieved_enum_ids', 'raw_vector_db_retrieved_enum_count', 'raw_vector_db_query_count', 'raw_vector_db_retrieved_count_by_query', 'raw_vector_db_query_limits', 'pre_prune_candidates_text', 'pre_prune_enum_ids', 'pre_prune_enum_count', 'post_prune_candidates_text', 'post_prune_enum_ids', 'post_prune_enum_count', 'expected_response', 'expected_enums', 'expected_enums_weights', 'agent_response', 'expert_score', 'expert_rationale', 'enum_relevance_score', 'execution_duration_ms', 'request_time', 'reranker_system_prompt', 'reranker_user_prompt', 'reranker_model', 'reranker_token_usage', 'reranker_raw_selected_ids', 'reranker_valid_selected_ids', 'reranker_invalid_selected_ids', 'reranker_unselect

# View data

In [0]:
display(df)

test_case_id trace_id user_query reranked_kb_context reranked_enum_ids raw_vector_db_retrieved_enum_ids raw_vector_db_retrieved_enum_count raw_vector_db_query_count raw_vector_db_retrieved_count_by_query raw_vector_db_query_limits pre_prune_candidates_text pre_prune_enum_ids pre_prune_enum_count post_prune_candidates_text post_prune_enum_ids post_prune_enum_count expected_response expected_enums expected_enums_weights agent_response expert_score expert_rationale enum_relevance_score execution_duration_ms request_time reranker_system_prompt reranker_user_prompt reranker_model reranker_token_usage reranker_raw_selected_ids reranker_valid_selected_ids reranker_invalid_selected_ids reranker_unselected_context_ids reranker_selection_status reranker_selection_violations main_agent_prompt_hash daily_banking_agent_prompt_hash agents_called tools_called query_scope reranker_selected_empty trace_invariant_violations kb_version user_query_en expected_response_en agent_response_en categories_list reranked_enums_kb_sk reranked_enums_kb_en post_prune_candidates_kb_sk post_prune_candidates_kb_en error error_type error_message raw_output_text case_scope expected_reference_looks_wrong expected_reference_issue_description optimal_enum_selection expected_answer_summary_with_optimal_context extra_or_distracting_enums unavailable_facts_in_selected_context missing_facts hallucinated_claims retrieved_pool_inadequacy_identified retrieved_pool_inadequacy_description retrieval_improvement_suggestion reranker_improvement_suggestion agent_improvement_suggestion kb_improvement_suggestion test_case_improvement_suggestion overall_explanation query_clarity_score query_clarity_reasoning selection_semantic_relevance_score selection_semantic_relevance_reasoning selected_context_sufficiency_score selected_context_sufficiency_reasoning optimal_retrieved_context_adequacy_score optimal_retrieved_context_adequacy_reasoning answer_expected_alignment_score answer_expected_alignment_reasoning answer_groundedness_score answer_groundedness_reasoning language_compliance_score language_compliance_reasoning Expected_answer_summary_with_optimal_context _csv_record_number _csv_load_warning Test case 512 tr-3f31a1fc540484398cfab33b318fa7f9 Potrebujem niečo riešiť ohľadom účtu, môžem napísať priamo svojmu poradcovi? [KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]
WRITE_MESSAGE: **Informácie o tom, ako získať kontakt na svojho konzultanta/poradcu:**\nMeno vášho konzultanta aj adresu jeho pobočky nájdete v aplikácii George. Stačí na hlavnej stránke v prehľade produktov prejsť na spodnú časť obrazovky a kliknúť na možnosť Kontakty.\nZobrazí sa vám meno konzultanta spolu s adresou pobočky na ktorej pôsobí.\nAk ho chcete kontaktovať, môžete mu priamo z Georgea odoslať správu cez možnosť Poslať správu. \nRovnako si s ním môžete dohodnúť stretnutie prostredníctvom možnosti Dohodnúť stretnutie.\n**Informácie o tom, ako napísať konzultantovi/poradcovi správu:**\nAk chcete napísať správu svojmu konzultantovi, stačí na hlavnej stránke v prehľade produktov prejsť na spodnú časť obrazovky a kliknúť na možnosť Kontakty. Otvorí sa vám stránka s informáciami o vašom konzultantovi, kde nájdete aj jeho meno a tlačidlo Poslať správu.\nNásledne si zo zoznamu dostupných tém vyberiete oblasť, o ktorej chcete písať, napíšete svoju otázku, prípadne priložíte súbor -- a hotovo, správu môžete odoslať.\nAk máte v Georgeovi povolené upozornenia, o odpovedi od konzultanta sa dozviete hneď, ako vám ju pošle.\n**Informácie o tom, ako odpovedať na správu doručenú od konzultanta/poradcu: **\nAk chcete odpovedať na správu, ktorá vám prišla od konzultanta, stačí na hlavnej stránke v prehľade produktov prejsť na spodnú časť obrazovky a kliknúť na možnosť Kontakty. V sekcii Doručené správny vyberte možnosť Messenger. Tu sa zobrazia vaše doručené správy, kde kliknutím na konkrétnu správu môžete zaslať odpoveď. \n
CALL_BRANCH_AUTHORISED: **Ako zatelefonovať poradcovi do banky cez Georgea**\nV súčasnosti nie je možné volať

## Queries sent to the KB search tool

In [0]:
def extract_tool_info(cell):
    if pd.isna(cell):
        return pd.Series([None, [], 0])

    try:
        data = json.loads(cell) if isinstance(cell, str) else cell

        if isinstance(data, list) and len(data) > 0:
            first = data[0]
            tool_name = first.get("name")
            queries = first.get("inputs", {}).get("queries", [])
            return pd.Series([tool_name, queries, len(queries)])

    except Exception:
        pass

    return pd.Series([None, [], 0])

df[["tool_name", "kb_search_queries", "kb_search_queries_size"]] = (
    df["tools_called"].apply(extract_tool_info)
)

In [0]:
tid = 1
df[df["test_case_id"] == f"Test case {tid}"][["tool_name", "kb_search_queries_size"]]

,tool_name,kb_search_queries_size
0,knowledge_search,5


In [0]:
df["kb_search_queries_size"].describe()

count    5.000000
mean     4.000000
std      2.236068
min      0.000000
25%      5.000000
50%      5.000000
75%      5.000000
max      5.000000
Name: kb_search_queries_size, dtype: float64

## Error check

In [0]:
if "error" in df.columns:
    err_mask = df["error"].astype(str).str.lower().isin({"true", "1", "1.0"})
    err_rows = df[err_mask]
    print(f"rows with error: {len(err_rows)}")
    if len(err_rows):
        cols = [c for c in ("test_case_id", "error", "error_message", "_csv_load_warning") if c in err_rows.columns]
        display(err_rows[cols].head(10))
else:
    print("no 'error' column")


rows with error: 0


## Weighted average + pass flag

In [0]:
for dim in DIMENSION_WEIGHTS:
    col = f"{dim}_score"
    if col not in df.columns:
        raise KeyError(f"expected dimension column '{col}' missing from checkpoint")
    df[col] = pd.to_numeric(df[col], errors="coerce")

weight_sum = sum(DIMENSION_WEIGHTS.values())
df["weighted_avg"] = sum(
    df[f"{dim}_score"] * w for dim, w in DIMENSION_WEIGHTS.items()
) / weight_sum

df["pass"] = df["weighted_avg"] >= PASS_THRESHOLD

# Judge-scored total (sum of the 8 judge dimension scores, range 0-16).
# Not asked of the judge — computed here.
df["total_score_judge"] = sum(df[f"{dim}_score"] for dim in DIMENSION_WEIGHTS)

print(f"pass rate:        {df['pass'].mean():.1%}  ({int(df['pass'].sum())}/{len(df)})")
print(f"weighted_avg     mean={df['weighted_avg'].mean():.3f}  median={df['weighted_avg'].median():.3f}")
print(f"total_score_judge mean={df['total_score_judge'].mean():.2f}  max possible = {2 * len(DIMENSION_WEIGHTS)}")
print()
print("per-dimension score means:")
for dim in DIMENSION_WEIGHTS:
    col = f"{dim}_score"
    print(f"  {dim:<24} mean={df[col].mean():.3f}  0-rate={(df[col]==0).mean():.1%}  2-rate={(df[col]==2).mean():.1%}")


pass rate:        69.6%  (337/484)
weighted_avg     mean=1.496  median=1.739
total_score_judge mean=12.58  max possible = 16

per-dimension score means:
  query_clarity            mean=1.841  0-rate=0.4%  2-rate=84.3%
  case_scope_fit           mean=1.954  0-rate=0.0%  2-rate=95.2%
  selection_semantic_relevance mean=1.451  0-rate=19.4%  2-rate=64.5%
  selected_context_sufficiency mean=1.449  0-rate=25.2%  2-rate=70.0%
  optimal_retrieved_context_adequacy mean=1.509  0-rate=22.5%  2-rate=73.3%
  answer_expected_alignment mean=1.219  0-rate=23.3%  2-rate=45.2%
  answer_groundedness      mean=1.197  0-rate=24.2%  2-rate=43.8%
  language_compliance      mean=1.959  0-rate=1.2%  2-rate=96.9%


## Per-dimension score distribution (0/1/2)

In [0]:
dim_cols = [f"{d}_score" for d in DIMENSION_WEIGHTS]
dist = (
    df[dim_cols]
    .apply(lambda s: s.value_counts(dropna=False).reindex([0, 1, 2, np.nan]).fillna(0).astype(int))
    .T
    .rename(columns={0: "score_0", 1: "score_1", 2: "score_2", np.nan: "null"})
)
dist["total"] = dist.sum(axis=1)
display(dist)


,score_0,score_1,score_2,NaN,total
query_clarity_score,2,73,408,1,484
case_scope_fit_score,0,22,461,1,484
selection_semantic_relevance_score,94,77,312,1,484
selected_context_sufficiency_score,122,22,339,1,484
optimal_retrieved_context_adequacy_score,109,19,355,1,484
answer_expected_alignment_score,113,151,219,1,484
answer_groundedness_score,117,154,212,1,484
language_compliance_score,6,8,469,1,484


In [0]:
n = len(DIMENSION_WEIGHTS)
fig = make_subplots(
    rows=2, cols=(n + 1) // 2,
    subplot_titles=list(DIMENSION_WEIGHTS.keys()),
    shared_yaxes=True,
)
for i, dim in enumerate(DIMENSION_WEIGHTS):
    row = (i // ((n + 1) // 2)) + 1
    col = (i % ((n + 1) // 2)) + 1
    counts = df[f"{dim}_score"].value_counts().reindex([0, 1, 2]).fillna(0)
    fig.add_trace(
        go.Bar(x=[0, 1, 2], y=counts.values, name=dim, showlegend=False),
        row=row, col=col,
    )
fig.update_layout(height=520, title_text="Per-dimension score distribution (0/1/2)")
fig.show()


## Optimal selection vs retrieved vs expected

In [0]:
cols=["optimal_enum_selection", "expected_enums", "reranked_enum_ids"]
df[cols].dtypes

optimal_enum_selection    str
expected_enums            str
reranked_enum_ids         str
dtype: object

In [0]:
df[["optimal_enum_selection", "expected_enums", "reranked_enum_ids"]].head(20)

,optimal_enum_selection,expected_enums,reranked_enum_ids
0,"[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS"", ""SAVING@KIDS_WITHDRAWALS_AND_DEPOSITS""]"
1,"[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS"", ""SAVING@ABOUT""]"
2,"[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS""]"
3,"[""SAVING@INTEREST_RATES_AND_LIMITS""]","[""SAVING@INTEREST_RATES_AND_LIMITS""]","[""SAVING@INTEREST_RATES_AND_LIMITS"", ""SAVING@ABOUT"", ""SAVING@DEPOSIT_INTEREST_RATES""]"
4,[],"[""SAVING@INTEREST_RATES_AND_LIMITS""]",[]
5,"[""SAVING@INTEREST_RATES_AND_LIMITS""]","[""SAVING@INTEREST_RATES_AND_LIMITS""]","[""SAVING@INTEREST_RATES_AND_LIMITS""]"
6,"[""SAVING@APPLICATION"", ""SAVING@ABOUT""]","[""SAVING@APPLICATION""]","[""SAVING@APPLICATION"", ""SAVING@ABOUT""]"
7,"[""SAVING@APPLICATION"", ""SAVING@ABOUT""]","[""SAVING@APPLICATION""]","[""SAVING@APPLICATION"", ""SAVING@ABOUT""]"
8,"[""SAVING@APPLICATION"", ""SAVING@ABOUT""]","[""SAVING@APPLICATION""]","[""SAVING@APPLICATION"", ""SAVING@ABOUT""]"
9,"[""SAVING@ABOUT""]","[""SAVING@ABOUT""]","[""SAVING@ABOUT""]"


In [0]:
import json

cols = ["optimal_enum_selection", "expected_enums", "reranked_enum_ids"]

def safe_json_loads(val):
    if not val or not isinstance(val, str):
        return None
    return json.loads(val)

for c in cols:
    df[c] = df[c].apply(safe_json_loads)


In [0]:
# Exact comparison
optimal_strict_acc = (df["optimal_enum_selection"] == df["expected_enums"]).mean()
reranked_strict_acc = (df["reranked_enum_ids"] == df["expected_enums"]).mean()

print(optimal_strict_acc)
print(reranked_strict_acc)

0.5206611570247934
0.3119834710743802


In [0]:
def safe_set(val):
    return set(val) if val is not None else set()

optimal_set_acc = (
    df.apply(lambda row: safe_set(row["optimal_enum_selection"]) == safe_set(row["expected_enums"]), axis=1)
).mean()

reranked_set_acc = (
    df.apply(lambda row: safe_set(row["reranked_enum_ids"]) == safe_set(row["expected_enums"]), axis=1)
).mean()

print("--- Reranker ---")
print(f"LLM judge: {(optimal_set_acc)*100:.2f}%")
print(f"George AI: {(reranked_set_acc)*100:.2f}%")


--- Reranker ---
LLM judge: 52.27%
George AI: 31.40%


## Programmatic derivations — `retrieval_recall`, `selection_was_optimal`, `root_cause_category`

These are NOT asked of the judge. They are computed here from:
- the 8 judge dimension scores
- `expected_enums`, `retrieved_enum_ids`, `reranked_enum_ids` (set operations)
- `missing_enums_in_candidate_pool` / `missing_enums_not_in_pool` (derived from expected/retrieved/reranked ENUM IDs)

The precedence order for `root_cause_category` follows the pipeline: query → retriever → reranker → KB → answer → language.


In [0]:
def _parse_list(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str) or not x.strip():
        return []
    try:
        parsed = json.loads(x)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return parsed if isinstance(parsed, list) else []


# Parse JSON list columns once
for col in (
    "reranked_enum_ids", "expected_enums", "retrieved_enum_ids",
    "missing_enums", "missing_enums_in_candidate_pool", "missing_enums_not_in_pool",
    "extra_or_distracting_enums", "optimal_enum_selection",
):
    if col in df.columns:
        df[f"_{col}"] = df[col].apply(_parse_list)


# ── 1. retrieval_recall — fraction of expected_enums that made it into the candidate pool ──
def _recall(r):
    e = set(r.get("_expected_enums", []))
    if not e:
        return np.nan
    p = set(r.get("_retrieved_enum_ids", []))
    return len(e & p) / len(e)

df["retrieval_recall"] = df.apply(_recall, axis=1)

def _recall_score(v):
    if pd.isna(v): return np.nan
    if v >= 0.999: return 2
    if v > 0.0:   return 1
    return 0

df["retrieval_recall_score"] = df["retrieval_recall"].apply(_recall_score)

df["_expected_enums_missed_by_retriever"] = df.apply(
    lambda r: sorted(set(r.get("_expected_enums", [])) - set(r.get("_retrieved_enum_ids", []))),
    axis=1,
)
df["_expected_enums_missed_by_reranker"] = df.apply(
    lambda r: sorted(
        (set(r.get("_expected_enums", [])) & set(r.get("_retrieved_enum_ids", [])))
        - set(r.get("_reranked_enum_ids", []))
    ),
    axis=1,
)
df["_missing_enums"] = df.apply(
    lambda r: sorted(set(r.get("_expected_enums", [])) - set(r.get("_reranked_enum_ids", []))),
    axis=1,
)
df["_missing_enums_not_in_pool"] = df["_expected_enums_missed_by_retriever"]
df["_missing_enums_in_candidate_pool"] = df["_expected_enums_missed_by_reranker"]
df["missing_enums"] = df["_missing_enums"].apply(json.dumps)
df["missing_enums_not_in_pool"] = df["_missing_enums_not_in_pool"].apply(json.dumps)
df["missing_enums_in_candidate_pool"] = df["_missing_enums_in_candidate_pool"].apply(json.dumps)

# ── 2. selection_was_optimal — reranker selection matches expected, no irrelevant additions ──
df["selection_was_optimal"] = df.apply(
    lambda r: set(r.get("_reranked_enum_ids", [])) == set(r.get("_expected_enums", []))
    if r.get("_expected_enums", []) else np.nan,
    axis=1,
)

# ── 3. root_cause_category — pipeline-ordered decision tree ──
def _root_cause(r) -> str:
    # If judge hit an error on this row, flag it
    if r.get("error") == True or str(r.get("error", "")).lower() == "true":
        return "judge_error"

    # Language drift is its own bucket (orthogonal to retrieval/KB)
    if r.get("language_compliance_score") == 0:
        return "language_mismatch"

    # Query clarity gates everything else
    if r.get("query_clarity_score") == 0:
        return "query_ambiguous"
    if r.get("case_scope_fit_score") == 0:
        return "scope_or_test_case_issue"

    # Retriever failure first — missing ENUM was never in the pool
    if r.get("retrieval_recall_score", 2) <= 1:
        return "retriever_missed_enums"

    # Reranker picked something off-topic
    if r.get("selection_semantic_relevance_score") == 0:
        return "reranker_wrong_selection"

    # Even the best retrieved candidate pool cannot support the answer
    if r.get("optimal_retrieved_context_adequacy_score") == 0:
        return "candidate_pool_content_gap"

    # Selected context is missing or too shallow for the expected answer
    if r.get("selected_context_sufficiency_score") <= 1:
        return "selected_context_insufficient"

    # The bot added unsupported claims beyond selected context
    if r.get("answer_groundedness_score") <= 1:
        return "hallucination_or_ungrounded_answer"

    # Retrieval + KB look fine but the bot's answer is wrong anyway
    if r.get("answer_expected_alignment_score") <= 1:
        return "answer_generation_issue"

    # Mild language issue (mostly_slovak, score 1) after the big checks pass
    if r.get("language_compliance_score") == 1:
        return "language_mismatch"

    return "no_issue"

df["root_cause_category"] = df.apply(_root_cause, axis=1)

# ── Summary printouts ──
print("retrieval_recall_score distribution:")
print(df["retrieval_recall_score"].value_counts(dropna=False).sort_index())
print(f"  mean retrieval_recall (continuous): {df['retrieval_recall'].mean():.3f}")
print()
print(f"selection_was_optimal: {df['selection_was_optimal'].sum()}/{len(df)}  ({df['selection_was_optimal'].mean():.1%})")
print()
print("root_cause_category distribution:")
rc = df["root_cause_category"].value_counts()
display(rc.to_frame("count").assign(share=lambda x: (x["count"] / x["count"].sum()).round(3)))


## Programmatic triage — out-of-scope and reranker-empty rows

Two boolean columns flow in from the data-prep stage (see `skkb_001_data_preparation.ipynb`):

- **`query_scope`** — deterministic classifier on `agents_called` + `tools_called`:
  - `kb` — `daily_banking_agent` called `knowledge_search` (the only in-scope slice for this judge)
  - `banking_data` — `daily_banking_agent` called a `mock_banking_*` tool (account/card/transaction lookup — not a KB question)
  - `rule1_violation` — `daily_banking_agent` was routed to but answered **without** calling any tool (violates the "always ground in KB" rule)
  - `routed_to_main` — `daily_banking_agent` was never invoked (main_agent handled it — investment, trading, etc.)
  - `other_tools` — catch-all
- **`reranker_selected_empty`** — True iff `query_scope == "kb"` AND the reranker returned `[]` (the bot had to answer with no grounding at all).

Both are flags for triage, not inputs to the score. They should be excluded (`query_scope != "kb"`) or highlighted (`rule1_violation`, `reranker_selected_empty`) when reading dimension aggregates, because a KB-focused rubric will drag down scores on rows where there was nothing to retrieve.

In [0]:
# query_scope distribution — drives which rows actually belong to this eval
if "query_scope" in df.columns:
    print("query_scope distribution:")
    print(df["query_scope"].value_counts(dropna=False))
    print()
    in_scope = df["query_scope"] == "kb"
    print(f"in-scope for KB judge (query_scope == 'kb'): {in_scope.sum()}/{len(df)}")
    print(f"out-of-scope (filter before aggregating):   {(~in_scope).sum()}/{len(df)}")
else:
    print("no 'query_scope' column in checkpoint — add it to passthrough_columns in the YAML")


query_scope distribution:
query_scope
kb                 416
main_agent          39
banking_data        15
rule1_violation     13
                     1
Name: count, dtype: int64

in-scope for KB judge (query_scope == 'kb'): 416/484
out-of-scope (filter before aggregating):   68/484


In [0]:
# Rule-1 violations: daily_banking_agent answered without calling any tool.
# These SHOULD have been grounded in KB — every one is a bot-behavior bug.
if "query_scope" in df.columns:
    rule1 = df[df["query_scope"] == "rule1_violation"].copy()
    print(f"rule1_violation rows (agent answered with no tool call): {len(rule1)}")
    if len(rule1):
        cols = [c for c in (
            "test_case_id", "user_query", "user_query_en", "bot_response",
            "agents_called", "tools_called",
        ) if c in rule1.columns]
        display(rule1[cols])
else:
    print("no 'query_scope' column in checkpoint")


rule1_violation rows (agent answered with no tool call): 13


,test_case_id,user_query,user_query_en,bot_response,agents_called,tools_called
56,Test case 55,"Keď sa to zrušilo, aká je náhrada, ako to mám ...","Since it was cancelled, what is the alternativ...","Rozumiem, že niečo sa „zrušilo“ a hľadáte náhr...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
117,Test case 118,"Kam je lepšie dať peniaze, keď ich nechcem nec...",Where is it better to put money if I don’t wan...,"Rozumiem, čo riešite. Toto je ale presne ten t...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
175,Test case 179,"Aké sú reálne výhody debetnej karty, okrem pla...","What are the real benefits of a debit card, be...","Tu je pár praktických výhod debetnej karty, kt...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
249,Test case 257,Koľko stojí toto poistenie a ako sa platí?,How much does this insurance cost and how is i...,"Pomôžem rád, ale najprv potrebujem vedieť, **a...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
252,Test case 258,Kedy má zmysel životné poistenie a pre koho je...,When does life insurance make sense and who is...,"Životné poistenie má zmysel hlavne vtedy, keď ...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
253,Test case 259,"Kto je poistník a kto je poistený, aký je v to...",Who is the policyholder and who is insured—wha...,Vysvetlím to na jednoduchom príklade s životný...,"[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
280,Test case 286,Dá sa čerpať naraz alebo aj po častiach?,Can it be drawn down all at once or also in pa...,"Rozumiem to tak, že sa pýtate na úver alebo ko...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
355,Test case 375,Koľko budem mesačne splácať,How much will my monthly installment be?,"Aby som Vám vedel pomôcť, potrebujem ešte pár ...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
369,Test case 387,Čo znamená konsolidácia úverov,What does loan consolidation mean?,"Konsolidácia úverov znamená, že viac vašich ex...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]
375,Test case 395,Čo potrebujem k žiadosti,What do I need for the application?,"Rozumiem to tak, že sa pýtate: čo potrebujem k...","[""main_agent"", ""daily_banking_agent"", ""llm""]",[]


In [0]:
# Reranker returned [] on an in-scope KB query — bot answered with zero grounding.
# Flag for triage: either KB genuinely has no relevant fragment (KB gap) or the reranker
# filtered everything out incorrectly (reranker bug).
if "reranker_selected_empty" in df.columns:
    empty_sel = df[df["reranker_selected_empty"].astype(str).str.lower().isin({"true", "1", "1.0"})].copy()
    print(f"reranker_selected_empty rows (KB query, reranker picked 0): {len(empty_sel)}")
    if len(empty_sel):
        cols = [c for c in (
            "test_case_id", "user_query", "user_query_en", "bot_response",
            "retrieved_enum_ids", "reranked_enum_ids", "candidate_pool_content_gap_description",
        ) if c in empty_sel.columns]
        display(empty_sel[cols])
else:
    print("no 'reranker_selected_empty' column in checkpoint — add it to passthrough_columns in the YAML")


## Retriever vs reranker failures

The judge partitions every missing expected ENUM into:
- `missing_enums_in_candidate_pool` — the ENUM was in `retrieved_candidates_text` but the reranker didn't pick it (**reranker failure**).
- `missing_enums_not_in_pool` — the ENUM never surfaced from the retriever (**retriever failure**).

The same ENUM appearing often in one bucket vs the other tells you which team to send the bug to.

In [0]:
# Reranker selection precision / recall / F1 (reranked_enum_ids vs expected_enums).
# Retriever recall is already computed in the previous cell.
def _prf(retrieved, expected):
    r = set(retrieved); e = set(expected)
    if not e and not r:
        return (np.nan, np.nan, np.nan)
    tp = len(r & e)
    precision = tp / len(r) if r else np.nan
    recall = tp / len(e) if e else np.nan
    if precision and recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0 if (precision == 0 or recall == 0) else np.nan
    return (precision, recall, f1)


df[["enum_precision", "enum_recall", "enum_f1"]] = df.apply(
    lambda r: pd.Series(_prf(r.get("_reranked_enum_ids", []), r.get("_expected_enums", []))),
    axis=1,
)

print("reranker selection (reranked_enum_ids vs expected_enums):")
print(f"  precision mean={df['enum_precision'].mean():.3f}")
print(f"  recall    mean={df['enum_recall'].mean():.3f}")
print(f"  F1        mean={df['enum_f1'].mean():.3f}")
print()
print("retriever candidate pool (retrieved_enum_ids vs expected_enums):")
print(f"  recall    mean={df['retrieval_recall'].mean():.3f}   median={df['retrieval_recall'].median():.3f}")


reranker selection (reranked_enum_ids vs expected_enums):
  precision mean=0.541
  recall    mean=0.629
  F1        mean=0.504

retriever candidate pool (retrieved_enum_ids vs expected_enums):
  recall    mean=0.675   median=1.000


In [0]:
# Top-25 most-frequently-missed ENUMs, split by cause
def _top(counter, k=25):
    return pd.DataFrame(counter.most_common(k), columns=["enum_id", "count"])

reranker_miss = Counter()
retriever_miss = Counter()
any_miss = Counter()
if "_missing_enums_in_candidate_pool" in df.columns:
    for lst in df["_missing_enums_in_candidate_pool"]:
        for e in lst: reranker_miss[e] += 1
if "_missing_enums_not_in_pool" in df.columns:
    for lst in df["_missing_enums_not_in_pool"]:
        for e in lst: retriever_miss[e] += 1
if "_missing_enums" in df.columns:
    for lst in df["_missing_enums"]:
        for e in lst: any_miss[e] += 1

print(f"total missing-enum events: reranker={sum(reranker_miss.values())}  retriever={sum(retriever_miss.values())}  any={sum(any_miss.values())}")

if reranker_miss:
    print("\nTop 25 reranker misses (ENUM was in pool but not picked):")
    display(_top(reranker_miss))
if retriever_miss:
    print("\nTop 25 retriever misses (ENUM never entered the pool):")
    display(_top(retriever_miss))


total missing-enum events: reranker=22  retriever=157  any=179

Top 25 reranker misses (ENUM was in pool but not picked):


,enum_id,count
0,SAVING@DEPOSIT_ABOUT,2
1,ROUND_UP_SAVING_APPLICATION,2
2,OUTGOING_TRX_CLAIM,2
3,SAVING@DEPOSIT_APPLICATION,1
4,SELL_ORDER,1
5,DATEIO@CLAIMS,1
6,GIRO@STANDARD_FEES,1
7,GIRO@STANDARD_LIMITS,1
8,CARD_DEBIT@FEES,1
9,CARD_CREDIT@BENEFITS,1



Top 25 retriever misses (ENUM never entered the pool):


,enum_id,count
0,CARD_CREDIT@DRAWDOWN,3
1,SHOW_STANDING_SWEEP_ORDERS_V2,3
2,FRAUD@ABOUT,3
3,SAVING@PRODUCTINFO,3
4,LOAN@MORTGAGE_PRODUCTINFO,3
5,ROUND_UP_SAVINGS,3
6,LOAN@MORTGAGE_ABOUT,3
7,ACCOUNT_STATEMENT_LIST + ACCOUNT_STATEMENT_LIS...,3
8,APP_NOTIFICATIONS,3
9,BOOK_APPOINTMENT,3


In [0]:
if reranker_miss or retriever_miss:
    # Side-by-side bar chart
    top_union = list(dict.fromkeys(
        [e for e, _ in reranker_miss.most_common(15)]
        + [e for e, _ in retriever_miss.most_common(15)]
    ))
    rows = []
    for e in top_union:
        rows.append({"enum_id": e, "cause": "reranker_miss", "count": reranker_miss.get(e, 0)})
        rows.append({"enum_id": e, "cause": "retriever_miss", "count": retriever_miss.get(e, 0)})
    fig = px.bar(
        pd.DataFrame(rows),
        x="enum_id", y="count", color="cause", barmode="group",
        title="Top missed ENUMs split by cause",
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


## KB-gap rate and descriptions flagged by the judge

In [0]:
if "candidate_pool_content_gap_identified" in df.columns:
    kb_gap = df["candidate_pool_content_gap_identified"].astype(str).str.lower().isin({"true", "1", "1.0"})
    print(f"KB gap rate: {kb_gap.mean():.1%}  ({int(kb_gap.sum())}/{len(df)})")

    top_cols = ["test_case_id", "user_query", "root_cause_category", "candidate_pool_content_gap_description"]
    top_cols = [c for c in top_cols if c in df.columns]
    display(df.loc[kb_gap, top_cols].head(20))
else:
    print("no candidate_pool_content_gap_identified column")


## Language compliance (Slovak-only deployment)

In [0]:
if "language_compliance_score" in df.columns:
    lc = df["language_compliance_score"].astype(float)
    print(f"language_compliance score distribution:")
    print(lc.value_counts().sort_index())
    print()
    czech_rate = (lc < 2).mean()
    print(f"fraction of responses with any language-mismatch issue: {czech_rate:.1%}")
    cols = ["test_case_id", "user_query", "bot_response", "language_compliance_reasoning", "bot_language_issue_evidence"]
    cols = [c for c in cols if c in df.columns]
    display(df[lc < 2][cols].head(25))


## Judge-vs-expert validation (Spearman)

In [0]:
if "expert_score" in df.columns:
    df["expert_score"] = pd.to_numeric(df["expert_score"], errors="coerce")
    mask = df["expert_score"].notna() & df["weighted_avg"].notna()
    rho, pval = spearmanr(df.loc[mask, "weighted_avg"], df.loc[mask, "expert_score"])
    print(f"Spearman  (weighted_avg, expert_score) = {rho:.3f}   p = {pval:.3g}   n = {int(mask.sum())}")

    hover = ["test_case_id", "root_cause_category"] if "root_cause_category" in df.columns else ["test_case_id"]
    fig = px.scatter(
        df, x="weighted_avg", y="expert_score",
        hover_data=hover,
        title=f"Judge weighted-avg vs expert 1-10 score  (Spearman rho = {rho:.2f})",
    )
    fig.update_traces(marker=dict(size=6, opacity=0.6))
    fig.show()
else:
    rho, pval = float("nan"), float("nan")
    print("no expert_score column")


Spearman  (weighted_avg, expert_score) = 0.445   p = 8.12e-25   n = 483


## Improvement-suggestion excerpts (retrieval / bot / KB)

In [0]:
for col in ("retrieval_improvement_suggestion", "bot_improvement_suggestion", "kb_improvement_suggestion"):
    if col in df.columns:
        populated = df[df[col].fillna("").str.len() > 0]
        print(f"{col}: {len(populated)}/{len(df)} rows populated")
        show_cols = ["test_case_id", "root_cause_category", col] if "root_cause_category" in df.columns else ["test_case_id", col]
        display(populated[show_cols].head(15))


retrieval_improvement_suggestion: 110/484 rows populated


,test_case_id,root_cause_category,retrieval_improvement_suggestion
4,Test case 5,retriever_missed_enums,Add query rewrites and synonyms for Slovak phr...
20,Test case 19,retriever_missed_enums,Add robust Slovak intent patterns for age thre...
63,Test case 65,retriever_missed_enums,Tune retrieval for short Slovak queries like “...
70,Test case 70,retriever_missed_enums,Add query expansion and boosting for ATM withd...
97,Test case 97,retriever_missed_enums,Add query expansions for Moneyback-related int...
104,Test case 104,retriever_missed_enums,Ensure basic definitional intents retrieve the...
117,Test case 118,retriever_missed_enums,Add query rewrites and synonyms for generic in...
128,Test case 127,retriever_missed_enums,Broaden candidate recall for generic 'limitati...
142,Test case 143,language_mismatch,Add targeted synonyms for 'foreign-currency ac...
161,Test case 161,retriever_missed_enums,Improve recall for Slovak terms and variants l...


bot_improvement_suggestion: 351/484 rows populated


,test_case_id,root_cause_category,bot_improvement_suggestion
1,Test case 2,hallucination_or_ungrounded_answer,Tighten the answer prompt to avoid adding timi...
2,Test case 0,hallucination_or_ungrounded_answer,Tighten the answer prompt to: 1) include all k...
3,Test case 3,hallucination_or_ungrounded_answer,Tighten the answer prompt to: 1) include exact...
4,Test case 5,retriever_missed_enums,Add a Slovak language guard and prohibit fabri...
6,Test case 6,hallucination_or_ungrounded_answer,Tighten answer prompts to avoid adding unstate...
8,Test case 7,answer_generation_issue,When the user asks whether an action can be do...
9,Test case 11,answer_generation_issue,Adjust answer prompt to include key product pa...
10,Test case 9,hallucination_or_ungrounded_answer,Tighten the answer prompt: instruct the agent ...
17,Test case 17,hallucination_or_ungrounded_answer,Tighten the answer-generation guardrails to av...
19,Test case 20,answer_generation_issue,Adjust the answer prompt to include core produ...


kb_improvement_suggestion: 105/484 rows populated


,test_case_id,root_cause_category,kb_improvement_suggestion
3,Test case 3,hallucination_or_ungrounded_answer,Harmonize savings-rate details across SAVING@A...
4,Test case 5,retriever_missed_enums,Ensure the savings interest ENUM includes Slov...
6,Test case 6,hallucination_or_ungrounded_answer,"In SAVING@APPLICATION, add a clear prerequisit..."
23,Test case 21,answer_generation_issue,Resolve the inconsistency between DEPOSIT_APPL...
33,Test case 34,hallucination_or_ungrounded_answer,Enhance SAVING@DEPOSIT_ABOUT to explicitly sta...
38,Test case 36,hallucination_or_ungrounded_answer,"If accurate, expand SAVING@KIDS_INTEREST_RATES..."
53,Test case 51,hallucination_or_ungrounded_answer,Consider explicitly stating whether users can ...
61,Test case 62,hallucination_or_ungrounded_answer,Resolve the fee inconsistency by clearly stati...
70,Test case 70,retriever_missed_enums,Tag the ATM withdrawal code article with addit...
75,Test case 76,hallucination_or_ungrounded_answer,Expand the portfolio disposer article to list ...


## Worst cases and judge-expert disagreements

In [0]:
worst_cols = (
    ["test_case_id", "weighted_avg", "expert_score", "root_cause_category", "enum_f1"]
    + [f"{d}_score" for d in DIMENSION_WEIGHTS]
    + ["overall_explanation"]
)
worst_cols = [c for c in worst_cols if c in df.columns]

print("Bottom 10 by weighted_avg:")
display(df.nsmallest(10, "weighted_avg")[worst_cols])


Bottom 10 by weighted_avg:


,test_case_id,weighted_avg,expert_score,root_cause_category,enum_f1,query_clarity_score,case_scope_fit_score,selection_semantic_relevance_score,selected_context_sufficiency_score,optimal_retrieved_context_adequacy_score,answer_expected_alignment_score,answer_groundedness_score,language_compliance_score,overall_explanation
4,Test case 5,0.260870,6.0,retriever_missed_enums,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,The user’s question is ambiguous between accou...
97,Test case 97,0.347826,6.0,retriever_missed_enums,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,The user’s question is ambiguous but the expec...
142,Test case 143,0.347826,6.0,language_mismatch,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,The user asked which cards can be issued to a ...
176,Test case 177,0.347826,3.0,retriever_missed_enums,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,The user’s question is ambiguous and lacks the...
186,Test case 186,0.347826,6.0,retriever_missed_enums,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,The user’s question is ambiguous about the ima...
196,Test case 198,0.347826,6.0,retriever_missed_enums,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,"The user query is ambiguous, and no KB context..."
277,Test case 284,0.347826,0.0,language_mismatch,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,The user’s question is clear and KB-answerable...
322,Test case 336,0.347826,0.0,language_mismatch,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,The user asked a clear KB question about a dis...
328,Test case 346,0.347826,0.0,retriever_missed_enums,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,The user’s request is ambiguous and typically ...
339,Test case 357,0.347826,0.0,language_mismatch,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,"The user asked how to check regular transfers,..."


In [0]:
if "expert_score" in df.columns:
    judge_norm = df["weighted_avg"] / 2.0
    expert_norm = (df["expert_score"] - 1) / 9.0
    df["_judge_expert_gap"] = (judge_norm - expert_norm).abs()

    print("Top 10 by |judge - expert| gap:")
    display(df.nlargest(10, "_judge_expert_gap")[worst_cols + ["_judge_expert_gap"]])


Top 10 by |judge - expert| gap:


,test_case_id,weighted_avg,expert_score,root_cause_category,enum_f1,query_clarity_score,case_scope_fit_score,selection_semantic_relevance_score,selected_context_sufficiency_score,optimal_retrieved_context_adequacy_score,answer_expected_alignment_score,answer_groundedness_score,language_compliance_score,overall_explanation,_judge_expert_gap
286,Test case 291,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user asked about required income for mortg...,1.111111
294,Test case 302,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user’s question is clear and fits the KB s...,1.111111
298,Test case 305,2.0,0.0,no_issue,0.4,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user asked how to apply for a refinancing ...,1.111111
300,Test case 307,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user’s question is clear and KB-answerable...,1.111111
313,Test case 322,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user asked how to start sending money. The...,1.111111
321,Test case 337,2.0,0.0,no_issue,0.5,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,This is a straightforward KB process query. Th...,1.111111
326,Test case 342,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user asks how to get into the app faster; ...,1.111111
327,Test case 343,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,"The user asked how to set up quick access, whi...",1.111111
334,Test case 353,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user asked for loan application requiremen...,1.111111
341,Test case 360,2.0,0.0,no_issue,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,The user’s query is straightforward and falls ...,1.111111


## Weighted-avg histogram

In [0]:
fig = px.histogram(
    df, x="weighted_avg", nbins=20,
    title=f"Weighted-avg distribution  (pass threshold = {PASS_THRESHOLD}; pass rate = {df['pass'].mean():.1%})",
)
fig.add_vline(x=PASS_THRESHOLD, line_dash="dash", line_color="red", annotation_text="pass threshold")
fig.show()


## Export HTML report

In [0]:
os.makedirs(REPORT_DIR, exist_ok=True)

def _df_html(df_, title, max_rows=15):
    return f"<h3>{title}</h3>\n{df_.head(max_rows).to_html(index=False, escape=True, classes='tbl')}"

kb_gap_rate = float("nan")
if "candidate_pool_content_gap_identified" in df.columns:
    kb_gap_rate = df["candidate_pool_content_gap_identified"].astype(str).str.lower().isin({"true","1","1.0"}).mean()

lang_fail_rate = float("nan")
if "language_compliance_score" in df.columns:
    lang_fail_rate = (pd.to_numeric(df["language_compliance_score"], errors="coerce") < 2).mean()

retriever_recall_mean = df["retrieval_recall"].mean() if "retrieval_recall" in df.columns else float("nan")

summary_html = f"""
<h2>Summary</h2>
<ul>
  <li>Experiment: <b>{YAML_FILE_NAME}</b></li>
  <li>Rows evaluated: <b>{len(df)}</b></li>
  <li>Pass rate (weighted_avg &ge; {PASS_THRESHOLD}): <b>{df['pass'].mean():.1%}</b></li>
  <li>Weighted-avg mean: <b>{df['weighted_avg'].mean():.3f}</b></li>
  <li>Judge vs expert Spearman: <b>{rho:.3f}</b> (p = {pval:.3g})</li>
  <li>Retriever recall (candidate pool &supseteq; expected): <b>{retriever_recall_mean:.3f}</b></li>
  <li>Reranker selection F1 vs expected: <b>{df['enum_f1'].mean():.3f}</b></li>
  <li>KB-gap rate: <b>{kb_gap_rate:.1%}</b></li>
  <li>Language-compliance failures (Slovak deployment): <b>{lang_fail_rate:.1%}</b></li>
</ul>
"""

n = len(DIMENSION_WEIGHTS)
fig_dist = make_subplots(rows=2, cols=(n + 1) // 2, subplot_titles=list(DIMENSION_WEIGHTS.keys()), shared_yaxes=True)
for i, dim in enumerate(DIMENSION_WEIGHTS):
    row = (i // ((n + 1) // 2)) + 1
    col = (i % ((n + 1) // 2)) + 1
    counts = df[f"{dim}_score"].value_counts().reindex([0, 1, 2]).fillna(0)
    fig_dist.add_trace(go.Bar(x=[0, 1, 2], y=counts.values, name=dim, showlegend=False), row=row, col=col)
fig_dist.update_layout(height=520, title_text="Per-dimension score distribution (0/1/2)")

fig_hist = px.histogram(df, x="weighted_avg", nbins=20, title=f"Weighted-avg distribution (pass threshold = {PASS_THRESHOLD})")
fig_hist.add_vline(x=PASS_THRESHOLD, line_dash="dash", line_color="red")

fig_rc = None
if "root_cause_category" in df.columns:
    rc = df["root_cause_category"].fillna("(missing)").value_counts().reset_index()
    rc.columns = ["root_cause", "count"]
    fig_rc = px.bar(rc, x="root_cause", y="count", title="Root cause category distribution")
    fig_rc.update_layout(xaxis_tickangle=-30)

fig_missing = None
if reranker_miss or retriever_miss:
    top_union = list(dict.fromkeys(
        [e for e, _ in reranker_miss.most_common(15)]
        + [e for e, _ in retriever_miss.most_common(15)]
    ))
    rows = []
    for e in top_union:
        rows.append({"enum_id": e, "cause": "reranker_miss", "count": reranker_miss.get(e, 0)})
        rows.append({"enum_id": e, "cause": "retriever_miss", "count": retriever_miss.get(e, 0)})
    fig_missing = px.bar(pd.DataFrame(rows), x="enum_id", y="count", color="cause", barmode="group",
                         title="Top missed ENUMs split by cause (retriever vs reranker)")
    fig_missing.update_layout(xaxis_tickangle=-45)

fig_corr = None
if "expert_score" in df.columns:
    fig_corr = px.scatter(df, x="weighted_avg", y="expert_score", hover_data=["test_case_id"],
                          title=f"Judge weighted-avg vs expert (rho = {rho:.2f})")

figs_html = ""
for i, fig in enumerate([f for f in [fig_dist, fig_hist, fig_rc, fig_missing, fig_corr] if f is not None]):
    figs_html += fig.to_html(include_plotlyjs="cdn" if i == 0 else False, full_html=False)

worst_html = _df_html(df.nsmallest(15, "weighted_avg")[[c for c in worst_cols if c in df.columns]], "Bottom 15 by weighted_avg")
gap_html = ""
if "_judge_expert_gap" in df.columns:
    gap_html = _df_html(
        df.nlargest(15, "_judge_expert_gap")[[c for c in worst_cols + ["_judge_expert_gap"] if c in df.columns]],
        "Top 15 by |judge - expert| gap",
    )

kb_gap_html = ""
if "candidate_pool_content_gap_description" in df.columns:
    kg = df[df["candidate_pool_content_gap_description"].fillna("").str.len() > 0][["test_case_id", "user_query", "candidate_pool_content_gap_description"]]
    kb_gap_html = _df_html(kg, "KB-gap descriptions flagged by the judge", max_rows=25)

lang_fail_html = ""
if "language_compliance_score" in df.columns:
    lc = pd.to_numeric(df["language_compliance_score"], errors="coerce")
    cols = [c for c in ("test_case_id", "user_query", "bot_response", "language_compliance_reasoning", "bot_language_issue_evidence") if c in df.columns]
    lang_fail_html = _df_html(df[lc < 2][cols], "Language-compliance failures (bot responded non-Slovak)", max_rows=25)

suggestion_htmls = []
for col, title in [
    ("retrieval_improvement_suggestion", "Retrieval / retriever improvement suggestions"),
    ("bot_improvement_suggestion", "Bot / agent improvement suggestions"),
    ("kb_improvement_suggestion", "KB editorial improvement suggestions"),
]:
    if col in df.columns:
        show_cols = ["test_case_id", "root_cause_category", col] if "root_cause_category" in df.columns else ["test_case_id", col]
        b = df[df[col].fillna("").str.len() > 0][show_cols]
        if len(b):
            suggestion_htmls.append(_df_html(b, title, max_rows=25))

html = f"""<!doctype html>
<html><head><meta charset='utf-8'><title>SKKB - {YAML_FILE_NAME}</title>
<style>
body {{ font-family: -apple-system, Segoe UI, Helvetica, Arial, sans-serif; margin: 2rem; max-width: 1200px; }}
h1 {{ color: #333; }}
.tbl {{ border-collapse: collapse; font-size: 12px; }}
.tbl th, .tbl td {{ border: 1px solid #ccc; padding: 4px 8px; text-align: left; vertical-align: top; max-width: 600px; }}
.tbl th {{ background: #f4f4f4; }}
</style></head><body>
<h1>SKKB Evaluation - {YAML_FILE_NAME}</h1>
{summary_html}
{figs_html}
{kb_gap_html}
{lang_fail_html}
{''.join(suggestion_htmls)}
{worst_html}
{gap_html}
</body></html>
"""

Path(REPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(REPORT_PATH, "w") as f:
    f.write(html)

print(f"wrote report: {REPORT_PATH}")


## Clickable HTML report

The richer, tabbed HTML report (Summary / Test Cases with click-through / Dimensions / Root Causes / Triage / Missed ENUMs) is generated by a standalone script to keep the notebook tidy.

Run from the repo root, or from this notebook via `!`:

```bash
cd experiments/skkb/notebooks && python skkb_report.py \
    --yaml-name skkb_exp_001_baseline \
    --reasoning-effort medium \
    --suffix _test
```

Output: `reports/<yaml_name>/report_clickable<suffix>.html`. See `skkb_report.py --help` for custom paths.

In [0]:
!ls

checkpoints		skkb_001_baseline.ipynb		 skkb_checkpoint.py
config_nbs.py		skkb_001_data_preparation.ipynb  skkb_report.py
export_dbx_nbs.py	skkb_001_final_analysis.ipynb	 traces_1.min.csv
mlflow.db		skkb_001_import_traces.ipynb	 traces_1min.csv
mlflow_trace_export.py	skkb_001_results.ipynb
notes.md		skkb_001_results_viewer.ipynb


In [0]:
!ls python skkb_report.py

ls: cannot access 'python': No such file or directory
skkb_report.py


In [0]:
!python skkb_report.py \
    --yaml-name skkb_exp_001_baseline \
    --reasoning-effort medium \
    --suffix _test

Traceback (most recent call last):
  File "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/notebooks/skkb_report.py", line 2078, in <module>
    main()
  File "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/notebooks/skkb_report.py", line 2063, in main
    df = enrich(df)
         ^^^^^^^^^^
  File "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/notebooks/skkb_report.py", line 160, in enrich
    raise KeyError(f"missing dimension column: {col}")
KeyError: 'missing dimension column: case_scope_fit_score'


## (Optional) Persist enriched results as a Delta table

In [0]:
%
# Uncomment to persist
# out_table = f"prod_aut_grag00_lab_catalog.private_uc055_grag_db.evals_{YAML_FILE_NAME}_{REASONING_EFFORT}"
# spark.createDataFrame(df.drop(columns=[c for c in df.columns if c.startswith('_')])).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(out_table)
# print(f"wrote {out_table}")
